
# Análise direta dos JSONL (STAI) — por temperatura e estilo de vida

Este notebook:
- Busca **JSONL** (`stai_respostas.jsonl` ou `personas.jsonl`) nas pastas do projeto;
- Limita a **3.000 registros por temperatura** (0.6, 0.75, 0.9);
- Recalcula **STAI total** a partir dos 20 itens (com inversão padronizada);
- Gera **gráficos (PNG)** e **tabelas em imagem (PNG)**, prontos para apresentação/TCC;
- Salva em `analise_json/graficos/` e `analise_json/tabelas_img/`.


In [3]:

import os, re, json, math, glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9, 5.2)
plt.rcParams['savefig.dpi'] = 160
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

ROOT = Path(".")
OUT_ROOT = Path("analise_json")
GFX_DIR = OUT_ROOT / "graficos"
TABIMG_DIR = OUT_ROOT / "tabelas_img"
for d in (OUT_ROOT, GFX_DIR, TABIMG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [
    Path("."),
    Path("./saida_sintetico_one_situation"),
    Path("./saida_sintetico"),
]

REVERSE_STAI = {1, 2, 5, 8, 10, 11, 15, 16, 19}

def stai_sum_from_row(row) -> int:
    total = 0
    for i in range(1, 21):
        v = int(row[f"stai_{i}"])
        v = 1 if v < 1 else 4 if v > 4 else v
        if i in REVERSE_STAI:
            total += 5 - v
        else:
            total += v
    return int(total)

def classify_lifestyle_by_sed(sed_ratio: float) -> str:
    if pd.isna(sed_ratio):
        return "moderado"
    if sed_ratio <= 0.35:
        return "muito ativo"
    elif sed_ratio <= 0.40:
        return "ativo"
    elif sed_ratio <= 0.48:
        return "moderado"
    else:
        return "sedentário"

def table_to_image(df: pd.DataFrame, title: str, path: Path, scale=1.0, max_rows=20):
    df_show = df.head(max_rows).copy()
    nrows, ncols = df_show.shape
    fig_h = 1.2 + 0.35 * (nrows + 1)
    fig_w = 2 + 1.2 * ncols
    fig, ax = plt.subplots(figsize=(fig_w*scale, fig_h*scale))
    ax.axis('off')
    ax.set_title(title)
    tbl = ax.table(cellText=df_show.values,
                   colLabels=df_show.columns,
                   loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.0, 1.2)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_linewidth(0.8)
        else:
            cell.set_linewidth(0.4)
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


In [4]:

def find_jsonl():
    paths = []
    for root in SEARCH_ROOTS:
        for pat in ("**/stai_respostas.jsonl", "**/personas.jsonl"):
            paths.extend(root.glob(pat))
    uniq = sorted(set(paths))
    return [p for p in uniq if p.is_file()]

files = find_jsonl()
print("Arquivos JSONL encontrados:")
for p in files:
    print(" -", p)

if not files:
    raise SystemExit("Nenhum JSONL encontrado. Coloque 'stai_respostas.jsonl' ou 'personas.jsonl' nas SEARCH_ROOTS.")


Arquivos JSONL encontrados:
 - saida_sintetico\temp_0_6\personas.jsonl
 - saida_sintetico\temp_0_75\personas.jsonl
 - saida_sintetico\temp_0_9\personas.jsonl
 - saida_sintetico_one_situation\temp_0_6\stai_respostas.jsonl
 - saida_sintetico_one_situation\temp_0_75\stai_respostas.jsonl
 - saida_sintetico_one_situation\temp_0_9\stai_respostas.jsonl


In [5]:

rows = []
import re
for p in files:
    temp_guess = None
    m = re.search(r"temp[_-](\d+)[._](\d+)", str(p).replace("\\","/"))
    if m:
        temp_guess = float(f"{m.group(1)}.{m.group(2)}")
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            out = {}
            out["id_persona"]   = rec.get("id") or rec.get("id_persona") or rec.get("persona_id")
            out["temperature"]  = rec.get("temperature", temp_guess)
            out["lifestyle"]    = rec.get("lifestyle") or rec.get("estilo") or rec.get("estilo_vida")
            out["situacao"]     = rec.get("situacao") or rec.get("situation") or rec.get("cenario")
            out["comentario"]   = rec.get("comentario") or rec.get("perfil") or rec.get("resumo") or ""
            for i in range(1, 21):
                out[f"stai_{i}"] = rec.get(f"stai_{i}")
                if out[f"stai_{i}"] is None and isinstance(rec.get("stai_items"), list) and len(rec.get("stai_items"))>=i:
                    out[f"stai_{i}"] = rec["stai_items"][i-1]
            out["sed_ratio"] = rec.get("sed_ratio")
            rows.append(out)

df = pd.DataFrame(rows)
print("Registros brutos:", len(df))

for i in range(1,21):
    df[f"stai_{i}"] = pd.to_numeric(df[f"stai_{i}"], errors="coerce")
df["temperature"] = pd.to_numeric(df["temperature"], errors="coerce")
df["sed_ratio"] = pd.to_numeric(df["sed_ratio"], errors="coerce")

df = df.dropna(subset=["temperature"])

mask_valid = np.ones(len(df), dtype=bool)
for i in range(1,21):
    mask_valid &= df[f"stai_{i}"].between(1,4, inclusive="both")
df = df[mask_valid].reset_index(drop=True)

if "lifestyle" not in df.columns or df["lifestyle"].isna().all():
    df["lifestyle"] = df["sed_ratio"].apply(classify_lifestyle_by_sed)
else:
    df["lifestyle"] = df["lifestyle"].fillna(df["sed_ratio"].apply(classify_lifestyle_by_sed))

print("Após limpeza:", len(df))


Registros brutos: 19984
Após limpeza: 19984


In [6]:

def capar_por_temperatura(df, n_por_temp=3000, chave_ordem="id_persona"):
    df = df.copy()
    if chave_ordem in df.columns:
        df = df.sort_values(["temperature", chave_ordem])
    else:
        df = df.sort_values(["temperature"]).reset_index(drop=True)
    capped = (
        df.groupby("temperature", group_keys=False)
          .apply(lambda g: g.head(n_por_temp))
          .reset_index(drop=True)
    )
    return capped

df = capar_por_temperatura(df, n_por_temp=3000)
print("Contagem por temperatura (capado):")
print(df["temperature"].value_counts().sort_index())


Contagem por temperatura (capado):
temperature
0.60    3000
0.75    3000
0.90    3000
Name: count, dtype: int64


C:\Temp\ipykernel_8476\1416431105.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(n_por_temp))


In [19]:
import pandas as pd

# ==========================
# Pivot base
# ==========================
pivot = (
    df.pivot_table(
        index="temperature",
        columns="lifestyle",
        values="stai_total",
        aggfunc=["mean", "count"]
    )
    .sort_index()
)

pivot.columns = [f"{stat}_{life}" for stat, life in pivot.columns]
pivot = pivot.reset_index()

# ==========================
# Tabela A: SÓ MÉDIAS
# ==========================
mean_cols = ["temperature"] + [c for c in pivot.columns if c.startswith("mean_")]
tab_A = pivot[mean_cols].copy()

tab_A = tab_A.rename(columns={"temperature": "Temperatura"} | {
    c: c.replace("mean_", "Média — ") for c in mean_cols if c != "temperature"
})

tab_A["Temperatura"] = tab_A["Temperatura"].astype(float).round(2)
tab_A = tab_A.round(2)

tab_A_path = TABIMG_DIR / "tabela_A_medias.png"
table_to_image(
    tab_A,
    "STAI por Temperatura × Lifestyle — Médias",
    tab_A_path,
    scale=1.0  # ajusta se quiser menor/maior
)
print("Tabela A (Médias) ->", tab_A_path)

# ==========================
# Tabela B: SÓ CONTAGENS (n)
# ==========================
count_cols = ["temperature"] + [c for c in pivot.columns if c.startswith("count_")]
tab_B = pivot[count_cols].copy()

tab_B = tab_B.rename(columns={"temperature": "Temperatura"} | {
    c: c.replace("count_", "n — ") for c in count_cols if c != "temperature"
})

tab_B["Temperatura"] = tab_B["Temperatura"].astype(float).round(2)
for c in tab_B.columns:
    if c != "Temperatura":
        tab_B[c] = tab_B[c].round(0)

tab_B_path = TABIMG_DIR / "tabela_B_contagens.png"
table_to_image(
    tab_B,
    "STAI por Temperatura × Lifestyle — Contagens (n)",
    tab_B_path,
    scale=1.0
)
print("Tabela B (n) ->", tab_B_path)

print("\n✅ Pronto: duas tabelas, sem DP.")


Tabela A (Médias) -> analise_json\tabelas_img\tabela_A_medias.png
Tabela B (n) -> analise_json\tabelas_img\tabela_B_contagens.png

✅ Pronto: duas tabelas, sem DP.


In [8]:
# ====== EDA compacta e bonita para slide (MMASH) ======
import numpy as np
import matplotlib.pyplot as plt

# --- dados (ajuste nomes se necessário) ---
x = df["sed_ratio"].astype(float)
y = df["STAI1"].astype(float)

# remove NaNs
m = ~(x.isna() | y.isna())
x = x[m].values
y = y[m].values

# --- regressão linear simples ---
coef = np.polyfit(x, y, 1)          # coef[0]=beta(slope), coef[1]=alpha(intercept)
y_hat = np.polyval(coef, x)

# --- intervalo de confiança simples (95%) da reta ---
n = len(x)
x_mean = x.mean()
s_err = np.sqrt(np.sum((y - y_hat)**2) / (n - 2))
x_line = np.linspace(x.min(), x.max(), 200)
y_line = np.polyval(coef, x_line)

# erro padrão da previsão
se_line = s_err * np.sqrt(1/n + (x_line - x_mean)**2 / np.sum((x - x_mean)**2))
ci = 1.96 * se_line  # ~95%

# --- plot compacto ---
plt.figure(figsize=(5.2, 3.2), dpi=300)
plt.scatter(x, y, s=18, alpha=0.75)
plt.plot(x_line, y_line, linewidth=2)
plt.fill_between(x_line, y_line-ci, y_line+ci, alpha=0.15)

plt.title("Sedentarismo vs Ansiedade (MMASH)", fontsize=12, pad=8)
plt.xlabel("sed_ratio (proporção sedentária)", fontsize=10)
plt.ylabel("STAI-1 (ansiedade-estado)", fontsize=10)

plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
plt.grid(alpha=0.25)
plt.tight_layout()

# salva em alta qualidade pro slide
plt.savefig("eda_mmash_sed_vs_stai1.png", dpi=300, bbox_inches="tight")
plt.savefig("eda_mmash_sed_vs_stai1.svg", bbox_inches="tight")  # opcional (melhor pra PowerPoint)
plt.show()

print("✅ Figura salva: eda_mmash_sed_vs_stai1.png / .svg")


KeyError: 'STAI1'

In [9]:

def plot_hist_stai(temp, g):
    fig, ax = plt.subplots()
    ax.hist(g["stai_total"].values, bins=20)
    ax.set_title(f"Histograma STAI — temp={temp}")
    ax.set_xlabel("STAI total")
    ax.set_ylabel("Frequência")
    p = GFX_DIR / f"hist_stai_temp_{str(temp).replace('.','_')}.png"
    fig.tight_layout(); fig.savefig(p); plt.close(fig)
    return p

def plot_box_stai_by_lifestyle(temp, g):
    order = ["muito ativo","ativo","moderado","sedentário"]
    data = [g.loc[g["lifestyle"]==lvl, "stai_total"].values for lvl in order]
    fig, ax = plt.subplots()
    ax.boxplot(data, labels=order, showfliers=False)
    ax.set_title(f"STAI por Lifestyle — temp={temp}")
    ax.set_xlabel("Lifestyle")
    ax.set_ylabel("STAI total")
    p = GFX_DIR / f"box_stai_lifestyle_temp_{str(temp).replace('.','_')}.png"
    fig.tight_layout(); fig.savefig(p); plt.close(fig)
    return p

def plot_bar_lifestyle_counts(temp, g):
    counts = g["lifestyle"].value_counts().sort_index()
    fig, ax = plt.subplots()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(f"Distribuição de Lifestyles — temp={temp}")
    ax.set_xlabel("Lifestyle")
    ax.set_ylabel("Contagem")
    p = GFX_DIR / f"bar_lifestyle_temp_{str(temp).replace('.','_')}.png"
    fig.tight_layout(); fig.savefig(p); plt.close(fig)
    return p

paths = []
for t, g in df.groupby("temperature"):
    paths.append(plot_hist_stai(t, g))
    paths.append(plot_box_stai_by_lifestyle(t, g))
    paths.append(plot_bar_lifestyle_counts(t, g))

print("Gráficos por temperatura gerados:")
for p in paths:
    print(" -", p)


C:\Temp\ipykernel_8476\3931869924.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=order, showfliers=False)
C:\Temp\ipykernel_8476\3931869924.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=order, showfliers=False)
C:\Temp\ipykernel_8476\3931869924.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=order, showfliers=False)


Gráficos por temperatura gerados:
 - analise_json\graficos\hist_stai_temp_0_6.png
 - analise_json\graficos\box_stai_lifestyle_temp_0_6.png
 - analise_json\graficos\bar_lifestyle_temp_0_6.png
 - analise_json\graficos\hist_stai_temp_0_75.png
 - analise_json\graficos\box_stai_lifestyle_temp_0_75.png
 - analise_json\graficos\bar_lifestyle_temp_0_75.png
 - analise_json\graficos\hist_stai_temp_0_9.png
 - analise_json\graficos\box_stai_lifestyle_temp_0_9.png
 - analise_json\graficos\bar_lifestyle_temp_0_9.png


In [10]:
# === Diagnóstico rápido antes da correlação ===
print("Temperaturas detectadas e contagens:", df["temperature"].value_counts().sort_index().to_dict())

has_sed = ("sed_ratio" in df.columns) and df["sed_ratio"].notna().any()
if has_sed:
    sed_counts = (
        df.dropna(subset=["sed_ratio"])
          .groupby("temperature")["sed_ratio"]
          .size()
          .sort_index()
          .to_dict()
    )
    print("Registros com sed_ratio por temperatura:", sed_counts)
else:
    print("Aviso: 'sed_ratio' indisponível nos JSONs — correlações/dispersões serão puladas.")

# === Correlação e dispersões (robusto a ausência de sed_ratio) ===
corr_rows = []

if has_sed:
    # Agrupar por temperatura (mesmo que tenha NaN), para ver tudo
    for t, g in df.groupby("temperature", dropna=False):
        gg = g.dropna(subset=["sed_ratio"]).copy()
        if len(gg) >= 3:
            r = float(np.corrcoef(gg["sed_ratio"].values, gg["stai_total"].values)[0, 1])
        else:
            r = np.nan
        corr_rows.append({"temperature": t, "pearson_sed_stai": r})

        # Scatter por temperatura, se tiver pontos
        if len(gg) > 0:
            fig, ax = plt.subplots()
            ax.scatter(gg["sed_ratio"], gg["stai_total"], s=8)
            ax.set_title(f"sed_ratio × STAI — temp={t}")
            ax.set_xlabel("sed_ratio")
            ax.set_ylabel("STAI total")
            p = GFX_DIR / f"scatter_sed_vs_stai_temp_{str(t).replace('.','_')}.png"
            fig.tight_layout(); fig.savefig(p); plt.close(fig)
            print("Gráfico ->", p)
else:
    # Sem sed_ratio: criar DF vazio com colunas esperadas para não quebrar
    corr_rows = []

# Montar DataFrame de correlações (sem quebrar se estiver vazio)
if len(corr_rows) == 0:
    print("Nenhuma correlação calculada (provável ausência de 'sed_ratio' nos JSONs).")
    corr_df = pd.DataFrame(columns=["temperature", "pearson_sed_stai"])
else:
    corr_df = pd.DataFrame(corr_rows)
    # Ordenar apenas se a coluna existir e o DF não estiver vazio
    if not corr_df.empty and "temperature" in corr_df.columns:
        corr_df = corr_df.sort_values("temperature").reset_index(drop=True)

# Salvar tabela em imagem, se houver algo
if not corr_df.empty:
    corr_path = TABIMG_DIR / "correlacoes_sed_stai.png"
    table_to_image(corr_df.round(3), "Correlação Pearson (sed_ratio × STAI) por Temperatura", corr_path, scale=1.0)
    print("Tabela ->", corr_path)

    # Gráfico de barras das correlações
    fig, ax = plt.subplots()
    ax.bar(corr_df["temperature"].astype(str), corr_df["pearson_sed_stai"].values)
    ax.set_title("Correlação sed_ratio × STAI por Temperatura")
    ax.set_xlabel("Temperatura")
    ax.set_ylabel("r de Pearson")
    p = GFX_DIR / "bar_correlacao_sed_stai_por_temp.png"
    fig.tight_layout(); fig.savefig(p); plt.close(fig)
    print("Gráfico ->", p)


Temperaturas detectadas e contagens: {0.6: 3000, 0.75: 3000, 0.9: 3000}
Aviso: 'sed_ratio' indisponível nos JSONs — correlações/dispersões serão puladas.
Nenhuma correlação calculada (provável ausência de 'sed_ratio' nos JSONs).


In [11]:

(OUT_ROOT / "README.txt").write_text(
"""
Saídas deste notebook:
- Imagens de gráficos: analise_json/graficos/
- Tabelas em PNG:      analise_json/tabelas_img/
- Dados lidos diretamente de JSONL e capados em 3.000 por temperatura.
- Métricas: média/DP de STAI, Cronbach alpha, distribuição por lifestyle,
  histogramas/boxplots por temperatura e, quando disponível, correlação sed_ratio × STAI.
""",
encoding="utf-8")
print("Pronto. Leia analise_json/README.txt")


Pronto. Leia analise_json/README.txt


In [12]:
# ============================================================
# EXPORTS: CSVs que ajudam MUITO na análise
# Gera tudo em analise_json/csv/
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd

CSV_DIR = OUT_ROOT / "csv"
CSV_DIR.mkdir(parents=True, exist_ok=True)

def _safe_cols(df, cols):
    return [c for c in cols if c in df.columns]

# --- 0) Dataframe limpo/capado completo (amigável) ---
cols_base = _safe_cols(
    df,
    ["id_persona", "temperature", "lifestyle", "situacao", "comentario", "sed_ratio", "stai_total"]
    + [f"stai_{i}" for i in range(1,21)]
)
df_clean = df[cols_base].copy()
df_clean.to_csv(CSV_DIR / "00_df_limpo_capado.csv", index=False, encoding="utf-8")
print("CSV ->", CSV_DIR / "00_df_limpo_capado.csv")

# --- 1) Resumo por temperatura (n, média, desvio, Cronbach) ---
resumo_temp.to_csv(CSV_DIR / "01_resumo_por_temperatura.csv", index=False, encoding="utf-8")
print("CSV ->", CSV_DIR / "01_resumo_por_temperatura.csv")

# --- 2) Resumo por temperatura × lifestyle (média/DP/contagem) ---
pivot = (
    df.pivot_table(
        index="temperature",
        columns="lifestyle",
        values="stai_total",
        aggfunc=["mean","std","count"]
    )
    .sort_index()
)
pivot.columns = [f"{a}_{b}" for a,b in pivot.columns]
pivot.reset_index().to_csv(CSV_DIR / "02_resumo_temp_x_lifestyle.csv", index=False, encoding="utf-8")
print("CSV ->", CSV_DIR / "02_resumo_temp_x_lifestyle.csv")

# --- 3) Distribuição de lifestyles por temperatura (contagens e percentuais) ---
dist_counts = (
    df.groupby(["temperature","lifestyle"])["stai_total"]
      .size()
      .reset_index(name="n")
      .sort_values(["temperature","lifestyle"])
)
dist_counts["pct_no_temp"] = (
    dist_counts.groupby("temperature")["n"].apply(lambda s: (100*s/s.sum()).round(2)).values
)
dist_counts.to_csv(CSV_DIR / "03_distribuicao_lifestyle_por_temp.csv", index=False, encoding="utf-8")
print("CSV ->", CSV_DIR / "03_distribuicao_lifestyle_por_temp.csv")

# --- 4) Médias por ITEM (1..20) por temperatura ---
cols_items = [f"stai_{i}" for i in range(1,21)]
medias_item_temp = (
    df.groupby("temperature")[cols_items].mean().reset_index().round(3)
)
medias_item_temp.to_csv(CSV_DIR / "04_medias_itens_por_temperatura.csv", index=False, encoding="utf-8")
print("CSV ->", CSV_DIR / "04_medias_itens_por_temperatura.csv")

# --- 5) Médias por ITEM por temperatura × lifestyle ---
medias_item_temp_life = (
    df.groupby(["temperature","lifestyle"])[cols_items].mean().round(3).reset_index()
)
medias_item_temp_life.to_csv(CSV_DIR / "05_medias_itens_por_temp_x_lifestyle.csv", index=False, encoding="utf-8")
print("CSV ->", CSV_DIR / "05_medias_itens_por_temp_x_lifestyle.csv")

# --- 6) Cronbach por temperatura (já temos). Agora por temperatura × lifestyle ---
def cronbach_alpha(mat: np.ndarray) -> float:
    if mat.shape[0] < 3:
        return np.nan
    k = mat.shape[1]
    item_var = mat.var(axis=0, ddof=1).sum()
    total_var = mat.sum(axis=1).var(ddof=1)
    if total_var <= 0:
        return np.nan
    return (k / (k - 1)) * (1 - (item_var / total_var))

rows_alpha = []
for (t, life), g in df.groupby(["temperature", "lifestyle"]):
    M = g[cols_items].to_numpy()
    rows_alpha.append({
        "temperature": t,
        "lifestyle": life,
        "n": len(g),
        "cronbach_alpha": float(cronbach_alpha(M))
    })
cron_life = pd.DataFrame(rows_alpha).sort_values(["temperature","lifestyle"])
cron_life.to_csv(CSV_DIR / "06_cronbach_por_temp_x_lifestyle.csv", index=False, encoding="utf-8")
print("CSV ->", CSV_DIR / "06_cronbach_por_temp_x_lifestyle.csv")

# --- 7) Correlação item×item (20×20) por temperatura (um CSV por temp) ---
for t, g in df.groupby("temperature"):
    if len(g) >= 3:
        corr = g[cols_items].corr().round(3)
        outp = CSV_DIR / f"07_correlacao_item_item_temp_{str(t).replace('.','_')}.csv"
        corr.to_csv(outp, encoding="utf-8")
        print("CSV ->", outp)

# --- 8) Amostras de comentários por temperatura × lifestyle (até 50 por célula) ---
amostras = (
    df_clean
      .sort_values(["temperature","lifestyle","id_persona"])
      .groupby(["temperature","lifestyle"], group_keys=False)
      .apply(lambda g: g[["id_persona","comentario","stai_total"]].head(50))
      .reset_index(drop=True)
)
amostras.to_csv(CSV_DIR / "08_amostras_comentarios_por_grupo.csv", index=False, encoding="utf-8")
print("CSV ->", CSV_DIR / "08_amostras_comentarios_por_grupo.csv")

# --- 9) (Opcional) Correlacao sed_ratio × STAI por temperatura + buckets de sedentarismo ---
has_sed = ("sed_ratio" in df_clean.columns) and df_clean["sed_ratio"].notna().any()
if has_sed:
    # 9.1 correlações
    corr_rows = []
    for t, g in df_clean.groupby("temperature"):
        gg = g.dropna(subset=["sed_ratio"])
        if len(gg) >= 3:
            r = float(np.corrcoef(gg["sed_ratio"].values, gg["stai_total"].values)[0, 1])
        else:
            r = np.nan
        corr_rows.append({"temperature": t, "pearson_sed_stai": r, "n_valid": len(gg)})
    corr_sed = pd.DataFrame(corr_rows).sort_values("temperature")
    corr_sed.to_csv(CSV_DIR / "09_correlacoes_sed_stai_por_temperatura.csv", index=False, encoding="utf-8")
    print("CSV ->", CSV_DIR / "09_correlacoes_sed_stai_por_temperatura.csv")

    # 9.2 quartis de sed_ratio e estatísticas de STAI por bucket
    dfq = df_clean.dropna(subset=["sed_ratio"]).copy()
    dfq["sed_bucket_q"] = dfq.groupby("temperature")["sed_ratio"] \
                             .transform(lambda s: pd.qcut(s, 4, labels=["Q1_baixo","Q2","Q3","Q4_alto"]))
    bucket_stats = (
        dfq.groupby(["temperature","sed_bucket_q"])
           .agg(
               n=("stai_total","size"),
               stai_mean=("stai_total","mean"),
               stai_std=("stai_total","std"),
               sed_mean=("sed_ratio","mean"),
               sed_std=("sed_ratio","std"),
           )
           .reset_index()
           .sort_values(["temperature","sed_bucket_q"])
           .round(3)
    )
    bucket_stats.to_csv(CSV_DIR / "10_stai_por_quartil_de_sed_ratio.csv", index=False, encoding="utf-8")
    print("CSV ->", CSV_DIR / "10_stai_por_quartil_de_sed_ratio.csv")

    # 9.3 dados de dispersão por temperatura (para replotar rápido se quiser)
    for t, g in dfq.groupby("temperature"):
        g[["id_persona","sed_ratio","stai_total"]].to_csv(
            CSV_DIR / f"11_scatter_data_temp_{str(t).replace('.','_')}.csv",
            index=False, encoding="utf-8"
        )
        print("CSV ->", CSV_DIR / f"11_scatter_data_temp_{str(t).replace('.','_')}.csv")
else:
    print("Aviso: sem 'sed_ratio' nos dados — CSVs 09/10/11 não foram gerados.")

# --- 10) Qualidade/validação por temperatura (itens fora da faixa e NAs) ---
def _flags_quality(df_):
    out = {}
    for i in range(1,21):
        s = df_[f"stai_{i}"]
        out[f"na_stai_{i}"] = int(s.isna().sum())
        out[f"out_range_stai_{i}"] = int(((s < 1) | (s > 4)).sum())
    return pd.Series(out)

qual_list = []
for t, g in df.groupby("temperature"):
    row = {"temperature": t, "n": len(g)}
    row = row | _flags_quality(g).to_dict()
    qual_list.append(row)
qual_df = pd.DataFrame(qual_list).sort_values("temperature")
qual_df.to_csv(CSV_DIR / "12_controle_qualidade_por_temperatura.csv", index=False, encoding="utf-8")
print("CSV ->", CSV_DIR / "12_controle_qualidade_por_temperatura.csv")

print("\n✅ Exports prontos em:", CSV_DIR.resolve())


CSV -> analise_json\csv\00_df_limpo_capado.csv
CSV -> analise_json\csv\01_resumo_por_temperatura.csv
CSV -> analise_json\csv\02_resumo_temp_x_lifestyle.csv
CSV -> analise_json\csv\03_distribuicao_lifestyle_por_temp.csv
CSV -> analise_json\csv\04_medias_itens_por_temperatura.csv
CSV -> analise_json\csv\05_medias_itens_por_temp_x_lifestyle.csv
CSV -> analise_json\csv\06_cronbach_por_temp_x_lifestyle.csv
CSV -> analise_json\csv\07_correlacao_item_item_temp_0_6.csv
CSV -> analise_json\csv\07_correlacao_item_item_temp_0_75.csv
CSV -> analise_json\csv\07_correlacao_item_item_temp_0_9.csv
CSV -> analise_json\csv\08_amostras_comentarios_por_grupo.csv
Aviso: sem 'sed_ratio' nos dados — CSVs 09/10/11 não foram gerados.
CSV -> analise_json\csv\12_controle_qualidade_por_temperatura.csv

✅ Exports prontos em: C:\Users\João\coleta_TCC2\analise_json\csv


C:\Temp\ipykernel_8476\195482815.py:108: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g[["id_persona","comentario","stai_total"]].head(50))


In [13]:
# ====== EDA compacta e bonita para slide (MMASH) ======
import numpy as np
import matplotlib.pyplot as plt

# --- dados (ajuste nomes se necessário) ---
x = df["sed_ratio"].astype(float)
y = df["STAI1"].astype(float)

# remove NaNs
m = ~(x.isna() | y.isna())
x = x[m].values
y = y[m].values

# --- regressão linear simples ---
coef = np.polyfit(x, y, 1)          # coef[0]=beta(slope), coef[1]=alpha(intercept)
y_hat = np.polyval(coef, x)

# --- intervalo de confiança simples (95%) da reta ---
n = len(x)
x_mean = x.mean()
s_err = np.sqrt(np.sum((y - y_hat)**2) / (n - 2))
x_line = np.linspace(x.min(), x.max(), 200)
y_line = np.polyval(coef, x_line)

# erro padrão da previsão
se_line = s_err * np.sqrt(1/n + (x_line - x_mean)**2 / np.sum((x - x_mean)**2))
ci = 1.96 * se_line  # ~95%

# --- plot compacto ---
plt.figure(figsize=(5.2, 3.2), dpi=300)
plt.scatter(x, y, s=18, alpha=0.75)
plt.plot(x_line, y_line, linewidth=2)
plt.fill_between(x_line, y_line-ci, y_line+ci, alpha=0.15)

plt.title("Sedentarismo vs Ansiedade (MMASH)", fontsize=12, pad=8)
plt.xlabel("sed_ratio (proporção sedentária)", fontsize=10)
plt.ylabel("STAI-1 (ansiedade-estado)", fontsize=10)

plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
plt.grid(alpha=0.25)
plt.tight_layout()

# salva em alta qualidade pro slide
plt.savefig("eda_mmash_sed_vs_stai1.png", dpi=300, bbox_inches="tight")
plt.savefig("eda_mmash_sed_vs_stai1.svg", bbox_inches="tight")  # opcional (melhor pra PowerPoint)
plt.show()

print("✅ Figura salva: eda_mmash_sed_vs_stai1.png / .svg")


KeyError: 'STAI1'

In [14]:
# ====== Gráfico EDA compacto para slide (auto-detecta STAI1) ======
import re
import numpy as np
import matplotlib.pyplot as plt

# 1) inspeciona colunas
print("Colunas disponíveis no df:")
print(list(df.columns))

# 2) normaliza nomes para procurar STAI-estado
def norm_name(s):
    return re.sub(r'[^a-z0-9]+', '_', str(s).lower()).strip('_')

cols_norm = {c: norm_name(c) for c in df.columns}

# 3) candidatos prováveis para STAI-estado
cands = []
for c, cn in cols_norm.items():
    # tenta pegar coisas tipo stai1 / stai_1 / stai_state / stai-state
    if ("stai" in cn) and (("1" in cn) or ("state" in cn)):
        cands.append(c)

if not cands:
    raise KeyError(
        "Não achei coluna de STAI-estado. "
        "Procure manualmente nas colunas acima e troque no código."
    )

stai1_col = cands[0]  # pega o melhor primeiro candidato
print("✅ Coluna STAI-estado detectada:", stai1_col)

# 4) checa sed_ratio
if "sed_ratio" not in df.columns:
    # tenta achar por aproximação
    sed_cands = [c for c, cn in cols_norm.items() if "sed_ratio" in cn or ("sed" in cn and "ratio" in cn)]
    if not sed_cands:
        raise KeyError("Não achei sed_ratio. Veja colunas acima.")
    sed_col = sed_cands[0]
else:
    sed_col = "sed_ratio"

print("✅ Coluna sedentarismo detectada:", sed_col)

# 5) dados
x = df[sed_col].astype(float)
y = df[stai1_col].astype(float)

m = ~(x.isna() | y.isna())
x = x[m].values
y = y[m].values

# 6) regressão linear + CI simples
coef = np.polyfit(x, y, 1)
y_hat = np.polyval(coef, x)

n = len(x)
x_mean = x.mean()
s_err = np.sqrt(np.sum((y - y_hat)**2) / (n - 2))
x_line = np.linspace(x.min(), x.max(), 200)
y_line = np.polyval(coef, x_line)

se_line = s_err * np.sqrt(1/n + (x_line - x_mean)**2 / np.sum((x - x_mean)**2))
ci = 1.96 * se_line

# 7) plot compacto e bonito
plt.figure(figsize=(5.2, 3.2), dpi=300)
plt.scatter(x, y, s=18, alpha=0.75)
plt.plot(x_line, y_line, linewidth=2)
plt.fill_between(x_line, y_line-ci, y_line+ci, alpha=0.15)

plt.title("Sedentarismo vs Ansiedade (MMASH)", fontsize=12, pad=8)
plt.xlabel("sed_ratio (proporção sedentária)", fontsize=10)
plt.ylabel("STAI-1 (ansiedade-estado)", fontsize=10)

plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig("eda_mmash_sed_vs_stai1.png", dpi=300, bbox_inches="tight")
plt.savefig("eda_mmash_sed_vs_stai1.svg", bbox_inches="tight")
plt.show()

print("✅ Figura salva: eda_mmash_sed_vs_stai1.png / .svg")


Colunas disponíveis no df:
['id_persona', 'temperature', 'lifestyle', 'situacao', 'comentario', 'stai_1', 'stai_2', 'stai_3', 'stai_4', 'stai_5', 'stai_6', 'stai_7', 'stai_8', 'stai_9', 'stai_10', 'stai_11', 'stai_12', 'stai_13', 'stai_14', 'stai_15', 'stai_16', 'stai_17', 'stai_18', 'stai_19', 'stai_20', 'sed_ratio', 'stai_total']
✅ Coluna STAI-estado detectada: stai_1
✅ Coluna sedentarismo detectada: sed_ratio


TypeError: expected non-empty vector for x

In [21]:
install scipy

SyntaxError: invalid syntax (1168798691.py, line 1)

In [22]:
# =========================================
# EDA COMPLETA — BASE SINTÉTICA STAI (SEM SCIPY)
# =========================================

import os, json, re
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------
# 0) CONFIGURAÇÃO
# ----------------------------
BASE_DIR = Path("saida_sintetico_one_situation")  # ajuste se necessário
TEMP_PREFIX = "temp_"
CSV_NAME = "stai_respostas.csv"
JSONL_NAME = "stai_respostas.jsonl"

OUT_DIR = BASE_DIR / "analise_eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 1) LOAD AUTOMÁTICO
# ----------------------------
def load_temp_folder(temp_dir: Path):
    csv_path = temp_dir / CSV_NAME
    jsonl_path = temp_dir / JSONL_NAME
    
    if csv_path.exists():
        df = pd.read_csv(csv_path)
    elif jsonl_path.exists():
        rows = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        df = pd.DataFrame(rows)
    else:
        raise FileNotFoundError(f"Não achei {CSV_NAME} nem {JSONL_NAME} em {temp_dir}")

    # inferir temperatura do nome da pasta temp_0_6 etc
    t_str = temp_dir.name.replace(TEMP_PREFIX, "").replace("_", ".")
    try:
        temperature = float(t_str)
    except:
        temperature = temp_dir.name
    
    df["temperature"] = temperature
    return df

temp_dirs = sorted([d for d in BASE_DIR.iterdir() if d.is_dir() and d.name.startswith(TEMP_PREFIX)])
if not temp_dirs:
    raise FileNotFoundError(f"Não encontrei pastas '{TEMP_PREFIX}*' dentro de {BASE_DIR}")

dfs = [load_temp_folder(d) for d in temp_dirs]
df = pd.concat(dfs, ignore_index=True)

print("✅ DataFrame carregado.")
print(df.head())
print("Colunas:", list(df.columns))

# ----------------------------
# 2) GARANTIR COLUNAS STAI E TOTAL
# ----------------------------
stai_cols = [c for c in df.columns if str(c).lower().startswith("stai_")]
stai_cols = [c for c in stai_cols if str(c).split("_")[-1].isdigit()]
stai_cols = sorted(stai_cols, key=lambda x: int(str(x).split("_")[-1]))

if len(stai_cols) < 20:
    print("\n⚠️ Aviso: não achei 20 colunas stai_1..stai_20. Achei:", stai_cols)

def stai_total_from_row(row):
    vals = pd.to_numeric(row[stai_cols], errors="coerce").values
    return np.nansum(vals)

if "stai_total" not in df.columns:
    df["stai_total"] = df.apply(stai_total_from_row, axis=1)

if "lifestyle" not in df.columns:
    raise KeyError("Coluna 'lifestyle' não encontrada.")
df["lifestyle"] = df["lifestyle"].astype(str).str.strip().str.lower()

# ----------------------------
# 3) DESCRITIVA GERAL
# ----------------------------
desc_geral = df["stai_total"].describe()
print("\n📌 STAI total — descritiva geral:")
print(desc_geral)

desc_geral.to_csv(OUT_DIR / "descritiva_geral_stai_total.csv")

# ----------------------------
# 4) TABELAS POR TEMPERATURA × LIFESTYLE
# ----------------------------
tab_mean = (
    df.groupby(["temperature", "lifestyle"])["stai_total"]
      .mean()
      .unstack("lifestyle")
      .sort_index()
      .round(2)
)

tab_count = (
    df.groupby(["temperature", "lifestyle"])["stai_total"]
      .size()
      .unstack("lifestyle")
      .sort_index()
)

print("\n📌 Médias STAI por Temperatura × Lifestyle:")
print(tab_mean)

print("\n📌 Contagens (n) por Temperatura × Lifestyle:")
print(tab_count)

tab_mean.to_csv(OUT_DIR / "medias_por_temp_lifestyle.csv")
tab_count.to_csv(OUT_DIR / "contagens_por_temp_lifestyle.csv")

# ----------------------------
# 5) COHEN'S D (efeito)
# ----------------------------
def cohens_d(a, b):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    na, nb = len(a), len(b)
    sa, sb = a.std(ddof=1), b.std(ddof=1)
    sp = np.sqrt(((na-1)*sa**2 + (nb-1)*sb**2) / (na+nb-2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan

# ----------------------------
# 6) BOOTSTRAP PARA DIFERENÇA DE MÉDIAS (ativo vs sedentário)
#    - Sem SciPy, mas dá IC e p-valor aproximado
# ----------------------------
def bootstrap_diff_means(a, b, n_boot=5000, seed=0):
    rng = np.random.default_rng(seed)
    a = np.asarray(a); b = np.asarray(b)
    diffs = []
    for _ in range(n_boot):
        sa = rng.choice(a, size=len(a), replace=True)
        sb = rng.choice(b, size=len(b), replace=True)
        diffs.append(sb.mean() - sa.mean())  # sedentário - ativo
    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_two = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return diffs.mean(), (ci_low, ci_high), p_two

tests_rows = []
temps_sorted = sorted(df["temperature"].unique())

for t in temps_sorted:
    sub = df[df["temperature"] == t]
    if not {"ativo", "sedentário"} <= set(sub["lifestyle"].unique()):
        continue

    ativo = sub[sub["lifestyle"] == "ativo"]["stai_total"].dropna()
    sedent = sub[sub["lifestyle"] == "sedentário"]["stai_total"].dropna()

    delta_mean, (ci_l, ci_h), p_boot = bootstrap_diff_means(ativo, sedent)
    d = cohens_d(ativo, sedent)

    tests_rows.append({
        "temperature": t,
        "mean_ativo": ativo.mean(),
        "mean_sedentario": sedent.mean(),
        "delta_sedent-ativo": delta_mean,
        "ci95_low": ci_l,
        "ci95_high": ci_h,
        "p_bootstrap": p_boot,
        "cohens_d": d
    })

tests_df = pd.DataFrame(tests_rows).round(4)
print("\n🧪 Ativo vs Sedentário (Bootstrap):")
print(tests_df)

tests_df.to_csv(OUT_DIR / "ativo_vs_sedentario_bootstrap.csv", index=False)

# ----------------------------
# 7) CRONBACH ALPHA POR TEMPERATURA
# ----------------------------
def cronbach_alpha(mat: np.ndarray) -> float:
    mat = mat.astype(float)
    mat = mat[~np.isnan(mat).any(axis=1)]
    if mat.shape[0] < 3:
        return np.nan
    k = mat.shape[1]
    item_var = mat.var(axis=0, ddof=1).sum()
    total_var = mat.sum(axis=1).var(ddof=1)
    if total_var <= 0 or k < 2:
        return np.nan
    return (k/(k-1)) * (1 - item_var/total_var)

alpha_rows = []
for t, g in df.groupby("temperature"):
    M = g[stai_cols].apply(pd.to_numeric, errors="coerce").to_numpy()
    alpha_rows.append({"temperature": t, "cronbach_alpha": cronbach_alpha(M)})

alpha_df = pd.DataFrame(alpha_rows).sort_values("temperature").round(4)
print("\n📌 Alpha de Cronbach por temperatura:")
print(alpha_df)

alpha_df.to_csv(OUT_DIR / "cronbach_por_temperatura.csv", index=False)

# ----------------------------
# 8) CORRELAÇÃO sed_ratio × stai_total (se existir)
# ----------------------------
if "sed_ratio" in df.columns:
    corr_rows = []
    for t, g in df.groupby("temperature"):
        x = pd.to_numeric(g["sed_ratio"], errors="coerce")
        y = pd.to_numeric(g["stai_total"], errors="coerce")
        m = ~(x.isna() | y.isna())
        if m.sum() < 3:
            continue
        r = np.corrcoef(x[m], y[m])[0, 1]
        corr_rows.append({"temperature": t, "pearson_r_sed_stai": r})

    corr_df = pd.DataFrame(corr_rows).round(4)
    print("\n📌 Correlação sed_ratio × STAI total:")
    print(corr_df)

    corr_df.to_csv(OUT_DIR / "correlacao_sed_ratio_stai.csv", index=False)
else:
    print("\nℹ️ sed_ratio não existe no sintético. Pulei correlação.")

print(f"\n✅ EDA finalizada. Outputs em: {OUT_DIR}")


✅ DataFrame carregado.
  id_persona  temperature  situacao lifestyle  stai_total  \
0    P007006          0.6  padrao_1  moderado          54   
1    P003796          0.6  padrao_1  moderado          55   
2    P005970          0.6  padrao_1  moderado          55   
3    P010796          0.6  padrao_1  moderado          54   
4    P001294          0.6  padrao_1  moderado          55   

                                          comentario  stai_1  stai_2  stai_3  \
0  Sinto uma leve ansiedade devido à mudança ines...       3       2       2   
1  A mudança de rota me deixou inquieto e um pouc...       3       2       3   
2  Sinto que a mudança de rota aumentou minha ans...       3       2       3   
3  A situação me deixou um pouco ansioso por cont...       3       2       2   
4  Sinto-me sobrecarregado e ansioso por não cons...       3       2       3   

   stai_4  ...  stai_11  stai_12  stai_13  stai_14  stai_15  stai_16  stai_17  \
0       3  ...        3        3        3       

In [23]:
# =========================================
# TABELAS FINAIS PARA SLIDE — EDA SINTÉTICO
# (SEM SCIPY)
# =========================================
import os, json, re
from pathlib import Path
import numpy as np
import pandas as pd

# -------- CONFIG --------
BASE_DIR = Path("saida_sintetico_one_situation")  # ajuste se necessário
TEMP_PREFIX = "temp_"
CSV_NAME = "stai_respostas.csv"
JSONL_NAME = "stai_respostas.jsonl"

OUT_DIR = BASE_DIR / "analise_eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# tenta detectar se sua função de imagem existe
HAS_TABLE_IMG = "table_to_image" in globals() and "TABIMG_DIR" in globals()
if HAS_TABLE_IMG:
    TABIMG_DIR.mkdir(parents=True, exist_ok=True)

# -------- LOAD AUTOMÁTICO --------
def load_temp_folder(temp_dir: Path):
    csv_path = temp_dir / CSV_NAME
    jsonl_path = temp_dir / JSONL_NAME
    
    if csv_path.exists():
        dfx = pd.read_csv(csv_path)
    elif jsonl_path.exists():
        rows = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        dfx = pd.DataFrame(rows)
    else:
        raise FileNotFoundError(f"Não achei {CSV_NAME} nem {JSONL_NAME} em {temp_dir}")

    t_str = temp_dir.name.replace(TEMP_PREFIX, "").replace("_", ".")
    dfx["temperature"] = float(t_str)
    return dfx

temp_dirs = sorted([d for d in BASE_DIR.iterdir() if d.is_dir() and d.name.startswith(TEMP_PREFIX)])
dfs = [load_temp_folder(d) for d in temp_dirs]
df = pd.concat(dfs, ignore_index=True)

# -------- GARANTIR STAI_COLS E TOTAL --------
stai_cols = [c for c in df.columns if str(c).lower().startswith("stai_")]
stai_cols = [c for c in stai_cols if str(c).split("_")[-1].isdigit()]
stai_cols = sorted(stai_cols, key=lambda x: int(str(x).split("_")[-1]))

if "stai_total" not in df.columns:
    df["stai_total"] = df[stai_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)

df["lifestyle"] = df["lifestyle"].astype(str).str.strip().str.lower()

# =====================================================
# TABELA 1) MÉDIAS STAI por Temperatura × Lifestyle
# =====================================================
tab_mean = (
    df.groupby(["temperature", "lifestyle"])["stai_total"]
      .mean()
      .unstack("lifestyle")
      .sort_index()
      .round(2)
      .reset_index()
      .rename(columns={"temperature": "Temperatura"})
)

print("\n📌 Tabela 1 — Médias STAI:")
display(tab_mean)

tab_mean.to_csv(OUT_DIR / "tabela_medias_temp_lifestyle.csv", index=False)

if HAS_TABLE_IMG:
    path_img_mean = TABIMG_DIR / "tabela_medias_temp_lifestyle.png"
    table_to_image(
        tab_mean,
        "STAI por Temperatura × Lifestyle — Médias",
        path_img_mean,
        scale=1.0
    )
    print("Imagem Tabela 1 ->", path_img_mean)

# =====================================================
# TABELA 2) CONTAGENS (n) por Temperatura × Lifestyle
# =====================================================
tab_count = (
    df.groupby(["temperature", "lifestyle"])["stai_total"]
      .size()
      .unstack("lifestyle")
      .sort_index()
      .reset_index()
      .rename(columns={"temperature": "Temperatura"})
)

print("\n📌 Tabela 2 — Contagens (n):")
display(tab_count)

tab_count.to_csv(OUT_DIR / "tabela_contagens_temp_lifestyle.csv", index=False)

if HAS_TABLE_IMG:
    path_img_count = TABIMG_DIR / "tabela_contagens_temp_lifestyle.png"
    table_to_image(
        tab_count,
        "STAI por Temperatura × Lifestyle — Contagens (n)",
        path_img_count,
        scale=1.0
    )
    print("Imagem Tabela 2 ->", path_img_count)

# =====================================================
# TABELA 3) SEDENTÁRIO vs ATIVO (Δ, IC95% bootstrap, d)
# =====================================================
def cohens_d(a, b):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    na, nb = len(a), len(b)
    sa, sb = a.std(ddof=1), b.std(ddof=1)
    sp = np.sqrt(((na-1)*sa**2 + (nb-1)*sb**2) / (na+nb-2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan

def bootstrap_diff_means(a, b, n_boot=5000, seed=0):
    rng = np.random.default_rng(seed)
    a = np.asarray(a); b = np.asarray(b)
    diffs = []
    for _ in range(n_boot):
        sa = rng.choice(a, size=len(a), replace=True)
        sb = rng.choice(b, size=len(b), replace=True)
        diffs.append(sb.mean() - sa.mean())  # sedentário - ativo
    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_two = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return diffs.mean(), ci_low, ci_high, p_two

rows = []
for t in sorted(df["temperature"].unique()):
    sub = df[df["temperature"] == t]
    if not {"ativo", "sedentário"} <= set(sub["lifestyle"].unique()):
        continue

    ativo = sub[sub["lifestyle"] == "ativo"]["stai_total"].dropna()
    sedent = sub[sub["lifestyle"] == "sedentário"]["stai_total"].dropna()

    delta, ci_l, ci_h, p_boot = bootstrap_diff_means(ativo, sedent)
    d = cohens_d(ativo, sedent)

    rows.append({
        "Temperatura": t,
        "Média Ativo": ativo.mean(),
        "Média Sedentário": sedent.mean(),
        "Δ (Sed - Ativo)": delta,
        "IC95% inferior": ci_l,
        "IC95% superior": ci_h,
        "p bootstrap": p_boot,
        "Cohen's d": d
    })

tab_comp = pd.DataFrame(rows).round(4)

print("\n📌 Tabela 3 — Sedentário vs Ativo:")
display(tab_comp)

tab_comp.to_csv(OUT_DIR / "tabela_comparacao_sedent_vs_ativo.csv", index=False)

if HAS_TABLE_IMG:
    path_img_comp = TABIMG_DIR / "tabela_comparacao_sedent_vs_ativo.png"
    table_to_image(
        tab_comp,
        "Comparação Sedentário vs Ativo (Δ, IC95%, d)",
        path_img_comp,
        scale=1.0
    )
    print("Imagem Tabela 3 ->", path_img_comp)

print(f"\n✅ Tabelas finais salvas em CSV em: {OUT_DIR}")
if HAS_TABLE_IMG:
    print(f"✅ E imagens salvas em: {TABIMG_DIR}")



📌 Tabela 1 — Médias STAI:


lifestyle,Temperatura,ativo,moderado,muito ativo,sedentário
0,0.60,50.18,49.99,51.15,50.20
1,0.75,49.94,49.84,52.50,50.44
2,0.90,50.04,49.96,50.30,50.32


Imagem Tabela 1 -> analise_json\tabelas_img\tabela_medias_temp_lifestyle.png

📌 Tabela 2 — Contagens (n):


lifestyle,Temperatura,ativo,moderado,muito ativo,sedentário
0,0.60,434,2405,13,332
1,0.75,413,2243,12,332
2,0.90,393,2273,10,324


Imagem Tabela 2 -> analise_json\tabelas_img\tabela_contagens_temp_lifestyle.png

📌 Tabela 3 — Sedentário vs Ativo:


,Temperatura,Média Ativo,Média Sedentário,Δ (Sed - Ativo),IC95% inferior,IC95% superior,p bootstrap,Cohen's d
0,0.60,50.1774,50.2048,0.0257,-0.5142,0.5657,0.9204,-0.0073
1,0.75,49.9370,50.4367,0.4986,0.0233,0.9775,0.0392,-0.1535
2,0.90,50.0382,50.3241,0.2911,-0.1769,0.7715,0.2284,-0.0864


Imagem Tabela 3 -> analise_json\tabelas_img\tabela_comparacao_sedent_vs_ativo.png

✅ Tabelas finais salvas em CSV em: saida_sintetico_one_situation\analise_eda
✅ E imagens salvas em: analise_json\tabelas_img
